In [1]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest, set_bokeh_output
set_bokeh_output(notebook=False)

from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
pair = 'CADCHF'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-23  0.95790  0.95845  0.94550  0.95260    81300
2010-09-24  0.95260  0.96180  0.94780  0.95980   103852
2010-09-27  0.96055  0.96420  0.95510  0.95655   115385
2010-09-28  0.95655  0.95765  0.94310  0.94775   126433
2010-09-29  0.94765  0.95425  0.94400  0.94620   115335
...             ...      ...      ...      ...      ...
2026-10-05  0.58075  0.58381  0.58046  0.58250    48951
2026-10-06  0.58249  0.58551  0.58186  0.58538    42990
2026-10-07  0.58540  0.58631  0.58206  0.58423    50257
2026-10-08  0.58424  0.58563  0.58294  0.58443    66525
2026-10-09  0.58444  0.58472  0.58121  0.58234    34325

[4185 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-23 08:00:00  0.95790  0.95845  0.95285  0.95370    19153
2010-09-23 12:00:00  0.95370  0.95455  0.94550  0.95

In [3]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 90

gap_look_ahead = 6

# nombre de candle au dessus/ en desous de key level après breakout
breakout_candles = 9

# nombre de jours à chercher key level
key_level_lookback_candle = 100

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles} candles au dessus du resistant level
breakout_candles_lookback = 30

start_trade_hour = 9
stop_trade_hour = 19

atr_multiplier = 0.7
min_touched = 3

In [4]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
for state in df_d1[df_d1["state_changed"]][["Close", "market_state"]].iterrows():
    print(state)
df_d1 = df_d1.loc[start_date : end_date]

(Timestamp('2020-12-02 00:00:00'), Close           0.69255
market_state        NaN
Name: 2020-12-02 00:00:00, dtype: object)
(Timestamp('2020-12-03 00:00:00'), Close           0.69245
market_state        NaN
Name: 2020-12-03 00:00:00, dtype: object)
(Timestamp('2020-12-04 00:00:00'), Close           0.69728
market_state        NaN
Name: 2020-12-04 00:00:00, dtype: object)
(Timestamp('2020-12-07 00:00:00'), Close           0.6955
market_state       NaN
Name: 2020-12-07 00:00:00, dtype: object)
(Timestamp('2020-12-08 00:00:00'), Close           0.69335
market_state        NaN
Name: 2020-12-08 00:00:00, dtype: object)
(Timestamp('2020-12-09 00:00:00'), Close           0.6932
market_state       NaN
Name: 2020-12-09 00:00:00, dtype: object)
(Timestamp('2020-12-10 00:00:00'), Close           0.69495
market_state        NaN
Name: 2020-12-10 00:00:00, dtype: object)
(Timestamp('2020-12-11 00:00:00'), Close           0.6964
market_state       NaN
Name: 2020-12-11 00:00:00, dtype: object)
(Times

In [5]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1,ATR
Date,,,,,,,,,,
2021-01-04 00:00:00,0.69378,0.69573,0.69310,0.69555,12960,False,False,BULLISH,BULLISH,0.000000
2021-01-04 04:00:00,0.69553,0.69625,0.69513,0.69527,9826,True,False,BULLISH,BULLISH,0.000000
2021-01-04 08:00:00,0.69525,0.69580,0.69311,0.69377,20631,False,False,BULLISH,BULLISH,0.000000
2021-01-04 12:00:00,0.69377,0.69405,0.69151,0.69293,23712,False,False,BULLISH,BULLISH,0.000000
2021-01-04 16:00:00,0.69296,0.69395,0.68858,0.68984,32860,False,False,BULLISH,BULLISH,0.000000
...,...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,0.58789,0.58839,0.58749,0.58791,6852,False,False,BULLISH,BULLISH,0.001281
2026-09-15 12:00:00,0.58792,0.58904,0.58735,0.58878,9087,False,False,BULLISH,BULLISH,0.001310
2026-09-15 16:00:00,0.58879,0.58880,0.58812,0.58812,4546,False,False,BULLISH,BULLISH,0.001265


In [20]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = Util.implement_market_structure_states_fixed_pivots(df_m15, n=structure_m15_window)

m15_daily_direction_list = []

index_list = []

indicator_atr_m15 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_m15._atr

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        # affect current market states of H4 candle because the result is market state of previous D1 candle
        m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

        # ajust to daylight saving time
        dst = pd.Timestamp(index2, tz=timezone).dst()
        if dst == timedelta(0):
            index2 = index2 - timedelta(hours=1)
            index_list.append(index2)
        else:
            index_list.append(index2)

df_m15['market_state_d1'] = m15_daily_direction_list

df_m15.index = index_list

df_m15

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,ATR,market_state_d1
2021-01-03 23:00:00,0.69378,0.69450,0.69310,0.69425,1856,False,True,BEARISH,0.000000,BULLISH
2021-01-03 23:15:00,0.69425,0.69472,0.69424,0.69428,451,False,False,BEARISH,0.000000,BULLISH
2021-01-03 23:30:00,0.69429,0.69438,0.69412,0.69423,595,False,False,BEARISH,0.000000,BULLISH
2021-01-03 23:45:00,0.69422,0.69422,0.69372,0.69403,767,False,True,BEARISH,0.000000,BULLISH
2021-01-04 00:00:00,0.69403,0.69450,0.69382,0.69413,914,True,False,BEARISH,0.000000,BULLISH
...,...,...,...,...,...,...,...,...,...,...
2026-09-15 23:00:00,0.58770,0.58809,0.58770,0.58803,238,False,False,BEARISH,0.000351,BULLISH
2026-09-15 23:15:00,0.58803,0.58809,0.58801,0.58808,144,False,False,BEARISH,0.000332,BULLISH
2026-09-15 23:30:00,0.58805,0.58816,0.58805,0.58813,170,False,False,BEARISH,0.000316,BULLISH
2026-09-15 23:45:00,0.58814,0.58820,0.58812,0.58820,147,False,False,BEARISH,0.000299,BULLISH


In [21]:
# using m15 market structure for entry
def apply_total_signal(df, start_trade_hour=9, stop_trade_hour=19):
    # Initialize the 'TotalSignal' column
    df_signal_list = []

    for i in range(0, len(df)):
        current_bar_date = df.index[i]
        signal = 0

        # short
        # d1 trending conditions
        c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
        if c_short_daily_direction:
            c_m15_ms_bearish = df['market_state'].iloc[i] == 'BEARISH'

            c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
            if c_time_trade and c_m15_ms_bearish:
                signal = 1

        # long
        c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
        if c_long_daily_direction:
            c_m15_ms_bullish = df['market_state'].iloc[i] == 'BULLISH'

            c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
            if c_time_trade and c_m15_ms_bullish:
                signal = 2

        df_signal_list.append(signal)

    return df_signal_list

df_m15['TotalSignal'] = apply_total_signal(df=df_m15, start_trade_hour=9, stop_trade_hour=19)

In [22]:
len(df_m15[df_m15.TotalSignal != 0])

31820

In [9]:
df_m15['pointpos'] = df_m15.apply(lambda row: Util.pointpos(row), axis=1)

In [10]:
start_date_chart = '2022-10-1 00:00'
end_date_chart = '2022-10-30 00:00'
dfpl = df_m15.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [13]:
class ChochStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    setup_duration_gap = 60
    stoploss_window = 150
    
    def init(self):
        self.current_day = None
        self.setup_duration = None
        self.order_placed_time = None

        super().init()

    def reset_order(self, current_bar_date):
        self.current_day = current_bar_date

        # delete placed order at the end of the day if not executed
        for order in self.orders:
            order_executed= False
            for trade in self.trades:
                if trade.tag == order.tag:
                    order_executed = True

            if order_executed == False:
                order.cancel()

        self.order_placed_time = None
        
    def get_tag(self):
        return f"{self.order_placed_time}"

    def get_highest_high(self):
            return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
        
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()

    def get_sl(self):
        return round(random.uniform(self.data['ATR'][-1]*1,self.data['ATR'][-1]*2.5), 5)
    
    def next(self):
        super().next()

        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date:
            self.reset_order(current_bar_date)

        if len(self.trades)==0:
            if self.data['TotalSignal'][-1]==1:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.sell(sl=sl1, tp=tp1, size=position_size)
                
            if self.data['TotalSignal'][-1]==2:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.buy(sl=sl1, tp=tp1, size=position_size)

In [23]:
start_date_chart = '2021-1-1 00:00'
end_date_chart = '2026-12-30 00:00'
global_dfopt = df_m15.loc[start_date_chart : end_date_chart]

bt = Backtest(global_dfopt, ChochStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run()

print(stats)
bt.plot()


Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    94.09357
Equity Final [$]                   8731.09324
Equity Peak [$]                    10536.8987
Return [%]                          -12.68907
Buy & Hold Return [%]                -15.2726
Return (Ann.) [%]                      -2.108
Volatility (Ann.) [%]                 8.77061
CAGR [%]                             -2.35348
Sharpe Ratio                         -0.24035
Sortino Ratio                        -0.32936
Calmar Ratio                         -0.09595
Alpha [%]                           -16.71578
Beta                                 -0.26366
Max. Drawdown [%]                    -21.9693
Avg. Drawdown [%]                    -1.09482
Max. Drawdown Duration      860 days 10:00:00
Avg. Drawdown Duration       31 days 01:07:00
# Trades                                  234
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p2113', ...)

In [24]:
# add some sl tp management

class RandomStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def get_sl(self):
        return round(random.uniform(self.data['ATR'][-1]*1,self.data['ATR'][-1]*2.5), 5)
    
    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 234:
            if self.data['market_state_d1'][-1] == 'BULLISH':
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = self.get_sl()
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [30]:
trades_list = []
while len(trades_list) != 600:
    random_index = randint(0, len(df_m15))
    candle = df_m15.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
        if 9 <= candle.name.hour and candle.name.hour <= 19:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=807: Broker canceled the order due to insufficient margin (equity=9703.38, margin_available=9703.38).
  warnings.warn(


Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    64.52222
Equity Final [$]                   9564.74182
Equity Peak [$]                   11222.58449
Return [%]                           -4.35258
Buy & Hold Return [%]                -15.2726
Return (Ann.) [%]                    -0.69628
Volatility (Ann.) [%]                 8.59014
CAGR [%]                             -0.77802
Sharpe Ratio                         -0.08106
Sortino Ratio                        -0.11445
Calmar Ratio                         -0.02966
Alpha [%]                            -6.96727
Beta                                  -0.1712
Max. Drawdown [%]                   -23.47404
Avg. Drawdown [%]                      -0.841
Max. Drawdown Duration     1791 days 15:45:00
Avg. Drawdown Duration       32 days 04:00:00
# Trades                                  233
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p3817', ...)

In [32]:
i = 0
expectancy_list = []
winrate_list = []
number_trades = []

while len(expectancy_list) < 500:
    trades_list = []
    while len(trades_list) != 600:
        random_index = randint(0, len(df_m15)-1)
        candle = df_m15.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
            if 9 <= candle.name.hour and candle.name.hour <= 19:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.0009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if 180 <= stats['# Trades'] and stats['# Trades'] <= 270:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        number_trades.append(stats['# Trades'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

0 1 0.050220507471271186 Done
2 2 -0.001521351082180007 Done
3 3 -0.06566401575893804 Done
4 4 0.06075720663367665 Done
5 5 -0.09612224510169602 Done
7 6 0.009802922382075295 Done
9 7 -0.04734929748127309 Done
11 8 -0.1092235254054327 Done
12 9 -0.049608930209815466 Done
13 10 0.01662147960172789 Done
14 11 -0.11524222453097407 Done
15 12 -0.04545623736332913 Done
18 13 0.010142832592884782 Done
22 14 -0.048656933916812924 Done
24 15 -0.11389735418414139 Done
25 16 -0.16728799584692672 Done
26 17 -0.001787512057427388 Done
27 18 0.02642987541424742 Done
28 19 0.002640076342742507 Done
31 20 -0.046539237573314914 Done
32 21 -0.03455648065241288 Done
35 22 -0.01258852796316783 Done
39 23 -0.01557646708352738 Done
40 24 -0.07956836385312262 Done
41 25 -0.02668661904793349 Done
42 26 -0.03241301349355602 Done
43 27 -0.022453827660389728 Done
45 28 -0.04994536491183281 Done
46 29 -0.05779246663549775 Done
47 30 -0.08390557692338503 Done
48 31 -0.02916422338686836 Done
49 32 0.05619289659355

In [33]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {-0.06797 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= -0.06797).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= -0.06797).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {33.21799 - mean_winrate}")

500
p95 = 0.04739036132305029
max = 0.12851216308562807
mean = -0.029739524904357485
min = -0.16861427761198983
gap = -0.03823047509564252
rank = 19.0
p_value = 0.810379241516966
500
p95 = 36.603147620092116
max = 40.97560975609756
mean = 31.984061928438823
min = 23.504273504273502
gap = 1.2339280715611771
